# 01 · Landing Zone

Gets the raw ENEM CSV into `data/landing/`. This step uses plain Python, no Spark.

- `USE_SAMPLE = True`: copies the 20-row sample from `sample_data/` (seconds, offline).
- `USE_SAMPLE = False`: downloads the official INEP microdata (~3 GB ZIP) and extracts the CSV.

In [ ]:
USE_SAMPLE = True
VERIFY_SSL = True   # INEP's certificate chain is sometimes incomplete; set False only if the download fails
REMOVE_ZIP = True   # delete the ZIP after extracting the CSV

In [ ]:
import shutil
import ssl
import urllib.request
import zipfile
from pathlib import Path

INEP_URL = "https://download.inep.gov.br/microdados/microdados_enem_2020.zip"
SAMPLE_CSV = Path.cwd() / "sample_data" / "sample_enem_2020.csv"
LANDING_DIR = Path.cwd() / "data" / "landing"
LANDING_CSV = LANDING_DIR / "enem_2020" / "MICRODADOS_ENEM_2020.csv"
ZIP_PATH = LANDING_DIR / "microdados_enem_2020.zip"

LANDING_CSV.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
if USE_SAMPLE:
    shutil.copyfile(SAMPLE_CSV, LANDING_CSV)
else:
    if not ZIP_PATH.exists():
        ctx = ssl.create_default_context()
        if not VERIFY_SSL:
            ctx.check_hostname = False
            ctx.verify_mode = ssl.CERT_NONE
        print("Downloading ENEM 2020 (~3 GB), this may take a while...")
        request = urllib.request.Request(INEP_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, context=ctx) as response, open(ZIP_PATH, "wb") as out:
            shutil.copyfileobj(response, out, 1024 * 1024)

    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        member = next(m for m in zip_file.namelist() if m.upper().endswith("MICRODADOS_ENEM_2020.CSV"))
        with zip_file.open(member) as source, open(LANDING_CSV, "wb") as target:
            shutil.copyfileobj(source, target)

    if REMOVE_ZIP:
        ZIP_PATH.unlink()

print(f"Landing ready: {LANDING_CSV} ({LANDING_CSV.stat().st_size / 1e6:.1f} MB)")

Peek at the raw file: `;` separated, ISO-8859-1 encoded, comma or dot as decimal separator.

In [ ]:
with open(LANDING_CSV, encoding="ISO-8859-1") as f:
    for _ in range(3):
        print(f.readline()[:300])